# Notebook 01 — Systems Thinking and Emergence

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 01 of 12  
**Time estimate:** 60 minutes

> The whole is more than the sum of its parts.
> Systems biology asks not what each gene does in isolation, but how their interactions
> produce robust, adaptive, and sometimes catastrophic collective behaviors.

---
## Step 1 — Motivation

Reductionism — the dominant paradigm in 20th-century biology — identified individual
genes, proteins, and pathways. But knowing that p53 is a tumor suppressor doesn't
explain why some cancers with wild-type p53 behave aggressively, or why the same
mutation causes different phenotypes in different genetic backgrounds.

Systems biology addresses this by modeling the *network* of interactions, not the
components in isolation.

---
## Step 2 — Intuition

**Emergence:** A property of a system that is not present in its components.
- Water is wet; H₂ and O₂ are not
- Consciousness is not in any single neuron
- Cell oscillation rhythms are not in any single protein

**Feedback loops** are the mechanistic basis of most emergent behavior:
- **Negative feedback:** output suppresses input → stabilization, homeostasis
  (thermostat, body temperature regulation, lac operon repression)
- **Positive feedback:** output amplifies input → bistability, commitment,
  irreversible switches (cell fate decisions, action potentials)

**Robustness:** The ability of a system to maintain function under perturbation.
Many biological systems are robust to component variation but fragile to specific
targeted failures — a hallmark of evolved network architecture.

---
## Step 3 — Biological Background

### Network Motifs

Alon (2002) showed that biological networks contain certain subgraph patterns
(**network motifs**) far more often than random networks of the same size/density.
These motifs are the "recurring circuit elements" of biological networks:

- **Autoregulation:** A gene regulates itself (most common motif in E. coli)
  - Negative autoregulation: speeds up response time, reduces noise
  - Positive autoregulation: slows response, creates bistability

- **Feedforward loop (FFL):** X activates Y and Z; Y activates Z
  - Coherent FFL: all arms activate — pulse generation, sign-sensitive delay
  - Incoherent FFL: arms conflict — pulse generator, fold-change detection

### The lac Operon as a Systems Biology Example

The *E. coli* lac operon (Monod & Jacob, Nobel 1965) is a classic gene regulatory
network with:
- **Negative feedback:** LacI repressor binds operator → blocks transcription
- **Positive feedback:** cAMP-CRP activates transcription when glucose is absent
- **Logic gate behavior:** operon ON only when lactose present AND glucose absent
  (implementing AND logic at the molecular level)

### Design Principles in Biology

- **Robustness:** exact adaptation in bacterial chemotaxis (E-BARK model)
- **Modularity:** evolution of distinct functional modules (protein domains, pathways)
- **Bow-tie architecture:** cellular metabolism — many inputs, few core reactions, many outputs

---
## Step 4 — Mathematical Explanation

**Simple negative feedback ODE:**
$$\frac{dx}{dt} = \beta - \alpha x$$

Steady state: $x^* = \beta/\alpha$. Small perturbation $\delta x$:
$$\frac{d(\delta x)}{dt} = -\alpha \, \delta x \implies \delta x(t) = \delta x_0 \, e^{-\alpha t}$$

The perturbation decays exponentially — the system returns to steady state. This is
the mathematics of homeostasis.

**Positive feedback → bistability:**
$$\frac{dx}{dt} = \frac{x^n}{K^n + x^n} - \delta x$$

When $n > 1$ (cooperative Hill function), this can have **three fixed points** —
two stable (low and high $x^*$) separated by one unstable. This is a bistable switch:
the system sits in one of two states and can be pushed between them by a sufficiently
large perturbation.

In [ ]:
# Step 6 — Python: Negative feedback dynamics and bistability
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ---- 1. Negative feedback: response to step input ----
def neg_feedback(t, x, beta, alpha):
    return [beta - alpha * x[0]]

# Parameters: step up at t=0 (beta switches from 0 to 1)
t_span = (0, 20)
t_eval = np.linspace(0, 20, 500)
sol_fast = solve_ivp(neg_feedback, t_span, [0], t_eval=t_eval, args=(1.0, 1.0))
sol_slow = solve_ivp(neg_feedback, t_span, [0], t_eval=t_eval, args=(1.0, 0.2))

# ---- 2. Bistability: Hill function positive feedback ----
def bistable(x, n=4, K=0.5, delta=1.0):
    """dx/dt = x^n / (K^n + x^n) - delta*x"""
    return x**n / (K**n + x**n) - delta * x

x_range = np.linspace(0.01, 1.5, 500)
dxdt_n1 = [bistable(x, n=1) for x in x_range]
dxdt_n4 = [bistable(x, n=4) for x in x_range]

# Find fixed points numerically (where dx/dt crosses zero)
fp_n4 = []
for i in range(len(x_range) - 1):
    if dxdt_n4[i] * dxdt_n4[i+1] < 0:
        fp_n4.append(x_range[i])
print(f'Bistable fixed points (n=4): {[f"{fp:.3f}" for fp in fp_n4]}')
print(f'  Low stable state: {fp_n4[0]:.3f}')
print(f'  Unstable threshold: {fp_n4[1]:.3f}')
print(f'  High stable state: {fp_n4[2]:.3f}' if len(fp_n4) > 2 else '  (only 2 FPs found)')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: Negative feedback dynamics
ax = axes[0]
ax.plot(sol_fast.t, sol_fast.y[0], 'steelblue', lw=2, label='Fast (α=1.0)')
ax.plot(sol_slow.t, sol_slow.y[0], 'tomato', ls='--', lw=2, label='Slow (α=0.2)')
ax.axhline(1.0, color='black', ls=':', lw=0.8, label='Steady state (β/α)')
ax.set_xlabel('Time')
ax.set_ylabel('x(t)')
ax.set_title('A. Negative feedback\n(step input at t=0)')
ax.legend(fontsize=8)

# Panel B: Phase portrait — dx/dt vs x (monostable vs bistable)
ax = axes[1]
ax.plot(x_range, dxdt_n1, 'steelblue', lw=2, label='n=1 (monostable)')
ax.plot(x_range, dxdt_n4, 'tomato', lw=2, label='n=4 (bistable)')
ax.axhline(0, color='black', lw=0.8)
for fp in fp_n4:
    ax.axvline(fp, color='grey', ls='--', lw=0.8, alpha=0.6)
ax.fill_between(x_range, 0, dxdt_n4, where=np.array(dxdt_n4) > 0,
                  alpha=0.1, color='tomato')
ax.fill_between(x_range, 0, dxdt_n4, where=np.array(dxdt_n4) < 0,
                  alpha=0.1, color='steelblue')
ax.set_xlabel('x')
ax.set_ylabel('dx/dt')
ax.set_title('B. Bistability via positive feedback\n(Hill coefficient n=4 → 3 fixed points)')
ax.legend(fontsize=8)

# Panel C: Feedforward loop logic diagram (conceptual)
ax = axes[2]
ax.axis('off')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
# Draw FFL diagram
ax.annotate('', xy=(0.5, 0.75), xytext=(0.2, 0.85),
              arrowprops={'arrowstyle': '->', 'color': 'steelblue', 'lw': 2})
ax.annotate('', xy=(0.8, 0.45), xytext=(0.5, 0.75),
              arrowprops={'arrowstyle': '->', 'color': 'steelblue', 'lw': 2})
ax.annotate('', xy=(0.8, 0.45), xytext=(0.2, 0.85),
              arrowprops={'arrowstyle': '->', 'color': 'steelblue', 'lw': 2,
                          'connectionstyle': 'arc3,rad=0.3'})
ax.text(0.18, 0.87, 'X (signal)', fontsize=10, ha='center', fontweight='bold')
ax.text(0.50, 0.78, 'Y (TF)', fontsize=10, ha='center', fontweight='bold')
ax.text(0.80, 0.42, 'Z (output)', fontsize=10, ha='center', fontweight='bold')
ax.text(0.35, 0.84, 'activate', fontsize=8, color='steelblue', ha='center')
ax.text(0.68, 0.65, 'activate', fontsize=8, color='steelblue', ha='center')
ax.text(0.35, 0.60, 'activate\n(direct)', fontsize=7, color='steelblue', ha='center')
ax.set_title('C. Coherent feedforward loop (FFL)\nX→Y→Z, X→Z: sign-sensitive delay', fontsize=9)

plt.suptitle('Module 12 NB01: Systems Thinking and Emergence', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('systems_thinking.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Modify the negative feedback ODE to include a delay: what happens to the system
   dynamics (oscillations can emerge)?
2. Find all fixed points of the bistable system (n=4) numerically. Which are stable
   and which is unstable? (*Hint: sign of derivative at each fixed point.*)
3. Draw a feedback diagram for the lac operon. Label each interaction as activating
   or inhibiting. Is the net loop positive or negative feedback?
4. What is the biological function of negative autoregulation? Why does it speed
   up response time?

---
## Step 10 — Quiz

1. What is emergence? Give a biological example not mentioned in this notebook.
2. What distinguishes positive from negative feedback loops in terms of system behavior?
3. What is a network motif? Name two motifs and their biological functions.
4. What is the bow-tie architecture of metabolism?
5. What does robustness mean in systems biology?

---
## Step 12 — Reflection

> *[In one paragraph: explain why understanding a gene's sequence and expression level
> in isolation is insufficient to predict its phenotypic effect, using a specific
> example from the biological background section.]*

---
*Next: `02_graph_theory_for_biological_networks.ipynb`*